# More advanced concepts: Parallel computation and caching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/x-datascience-datacamp/datacamp-master/blob/main/01_advanced_sklearn/02-parallel_and_caching_with_joblib.ipynb)

```
Authors: Alexandre Gramfort
         Thomas Moreau
         Mathurin Massias
```

The aim of this notebook is:

  - to explain how parallel computation works within scikit-learn
  - how to cache certain computations to save computation time.

For this tutorial we will rely essentially on the [joblib package](https://joblib.readthedocs.io/).

In [ ]:
# Colab: clone the course repository and run from this notebook's folder
import os, subprocess, sys

if "google.colab" in sys.modules:
    repo = "/content/datacamp-master"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/x-datascience-datacamp/datacamp-master", repo], check=True)
    os.chdir(f"{repo}/01_advanced_sklearn")

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import os
from urllib.request import urlretrieve

url = ("https://archive.ics.uci.edu/ml/machine-learning-databases"
       "/adult/adult.data")
local_filename = os.path.basename(url)
if not os.path.exists(local_filename):
    print("Downloading Adult Census datasets from UCI")
    urlretrieve(url, local_filename)

In [ ]:
names = ("age, workclass, fnlwgt, education, education-num, "
         "marital-status, occupation, relationship, race, sex, "
         "capital-gain, capital-loss, hours-per-week, "
         "native-country, income").split(', ')    
data = pd.read_csv(local_filename, names=names)

y = data['income']
X_df = data.drop('income', axis=1)

In [ ]:
X_df.head()

In [ ]:
y.value_counts()

## Let's construct a full model with a ColumnTransformer

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

numeric_features = [c for c in X_df
                    if X_df[c].dtype.kind in ('i', 'f')]
categorical_features = [c for c in X_df
                        if X_df[c].dtype.kind not in ('i', 'f')]

pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown='ignore'), categorical_features),
        (StandardScaler(), numeric_features),
    ),
    RandomForestClassifier(
        max_depth=7, n_estimators=300, random_state=42
    )
)
pipeline

### How to run things in parallel in scikit-learn: The `n_jobs` parameter

In [ ]:
%%timeit -n1 -r2
cv_scores = cross_val_score(pipeline, X_df, y, scoring='roc_auc', cv=5)
print("CV score:", np.mean(cv_scores))

In [ ]:
%%timeit -n1 -r2
cv_scores = cross_val_score(pipeline, X_df, y, scoring='roc_auc', cv=5, n_jobs=5)
print("CV score:", np.mean(cv_scores))

In [ ]:
from sklearn.base import clone

pipeline_parallel_rf = clone(pipeline)
pipeline_parallel_rf.set_params(randomforestclassifier__n_jobs=-1)

In [ ]:
%%timeit -n1 -r2
cv_scores = cross_val_score(
    pipeline_parallel_rf, X_df, y, scoring='roc_auc',
    cv=5, n_jobs=1
)
print("CV score:", np.mean(cv_scores))

In [ ]:
%%timeit -n1 -r2
cv_scores = cross_val_score(pipeline_parallel_rf, X_df, y, scoring='roc_auc', cv=5, n_jobs=5)
print("CV score:", np.mean(cv_scores))

### How to write your own parallel code with joblib

Let's first look at a simple example:

In [ ]:
from joblib import Parallel, delayed
from math import sqrt

@delayed
def my_new_func(x):
    return 2*x

delayed(sqrt)

In [ ]:
delayed(sqrt)(4, b=2)

In [ ]:
import os
import time

def f(x):
    time.sleep(.5)
    return 2 * x, os.getpid()

print("Main process in ", os.getpid())
results = Parallel(n_jobs=2)(
    delayed(f)(i) for i in range(5)
)
print(results)
results, pids = list(zip(*results))
print(set(pids))
print(results)

Let's now do a full cross-validation in parallel:

In [ ]:
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

def _fit_score(model, X, y, train_idx, test_idx):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    model = clone(model)
    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)[:, 1]
    
    return roc_auc_score(y_test, y_score)

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_jobs = 5

cv = StratifiedKFold(n_splits=5)

scores = Parallel(n_jobs=n_jobs)(delayed(_fit_score)(
    pipeline_parallel_rf, X_df, y, train_idx, test_idx
) for train_idx, test_idx in cv.split(X_df, y))

print(np.mean(scores))

In [ ]:
# !pip install -U distributed
from distributed import Client
from joblib import parallel_config

client = Client()

with parallel_config(backend="dask"):
    cv_scores = cross_val_score(pipeline_parallel_rf, X_df, y, scoring='roc_auc', cv=5, n_jobs=-1)

print("CV score:", np.mean(cv_scores))
client.close()

### How about caching?

Something you want to avoid redoing again and again the same computations.
One classical solution to address this is called [function memoization](https://en.wikipedia.org/wiki/Memoization).

joblib offers a very trivial way to do using a simple Python decorator.

In [ ]:
from joblib import Memory

from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold

location = '__cache__'
mem = Memory(location, verbose=0)
mem.clear()  # in our case, make sure there is not left over cache from previous run

_fit_score_cached = mem.cache(_fit_score)

def evaluate_model():
    cv = StratifiedKFold(n_splits=5)
    cv_scores = Parallel(n_jobs=5)(delayed(_fit_score_cached)(
        pipeline_parallel_rf, X_df, y, train_idx, test_idx
    ) for train_idx, test_idx in cv.split(X_df, y))
    print(np.mean(cv_scores))

%timeit -n1 -r1 evaluate_model()

In [ ]:
%timeit -n1 -r1 evaluate_model()

In [ ]:
pipeline

The `Pipeline` object has a `memory` parameter. This allows to cache the computation by transformers inside the pipeline, for example to avoid rerunning the same preprocessing in a grid-search when tuning the classifier or regressor at the end of the pipeline (see the description of the parameter in the doc).

To go further you can also look how joblib can be used in combination with [dask-distributed](http://distributed.dask.org/en/stable/) to run computations across different machines or with [dask-jobqueue](http://jobqueue.dask.org/en/latest/) to use a cluster with a queuing system like `slurm`.

In [ ]:
from sklearn.pipeline import Pipeline
Pipeline?